### Seção 1: Importações de Bibliotecas


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, lit
from pyspark.sql.types import IntegerType, DecimalType
import os
import time
import psycopg2
from pyspark.sql.utils import AnalysisException

### Seção 2: Configuração e Inicialização


In [2]:
print("Iniciando a sessão Spark...")
spark = SparkSession.builder \
    .appName("Formula1_ETL_Silver_to_Gold_v1") \
    .getOrCreate()
print("Sessão Spark iniciada com sucesso!")

jdbc_hostname = os.getenv("DB_HOST", "db")
jdbc_port     = os.getenv("DB_PORT", "5432")
jdbc_database = os.getenv("DB_NAME", "f1database")
db_user       = os.getenv("DB_USER", "f1user")
db_password   = os.getenv("DB_PASSWORD", "1234")

jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}"
connection_properties = {
    "user": db_user, 
    "password": db_password, 
    "driver": "org.postgresql.Driver",
}

SILVER_TABLE = "ResultadosCorridas"
GOLD_SCHEMA = "gold"

Iniciando a sessão Spark...


Sessão Spark iniciada com sucesso!


### Seção 3: Conexão ao Banco de Dados (Verificação)

In [3]:
retries = 10
wait_seconds = 5
for i in range(retries):
    try:
        print("Tentando conectar ao banco de dados...")
        conn = psycopg2.connect(host=jdbc_hostname, dbname=jdbc_database, user=db_user, password=db_password, port=jdbc_port)
        conn.close()
        print("✅ Conexão com o banco de dados bem-sucedida!")
        break
    except psycopg2.OperationalError:
        print(f"⏳ Banco de dados não está pronto. Tentando novamente em {wait_seconds} segundos...")
        time.sleep(wait_seconds)
        if i == retries - 1:
            print("❌ Não foi possível conectar ao banco de dados. Abortando.")
            spark.stop()
            exit(1)

Tentando conectar ao banco de dados...
✅ Conexão com o banco de dados bem-sucedida!


### Seção 4: Leitura da Camada Silver

In [4]:
print(f"\nLendo tabela da Camada Silver: {SILVER_TABLE}")
try:
    df_silver = spark.read.jdbc(url=jdbc_url, table=SILVER_TABLE, properties=connection_properties)
    
    print("Aplicando CAST explícito nas colunas ID (chaves de negócio)...")
    df_silver = df_silver.withColumn("id_equipe", col("id_equipe").cast(IntegerType())) \
                          .withColumn("id_piloto", col("id_piloto").cast(IntegerType())) \
                          .withColumn("id_corrida", col("id_corrida").cast(IntegerType())) \
                          .withColumn("id_status", col("id_status").cast(IntegerType()))

    count_silver = df_silver.count()
    print(f"✅ Dados da Silver carregados. Total de registros LIDOS: {count_silver}")
    if count_silver == 0:
        print("⚠️ AVISO: A Camada Silver está vazia. O ETL GOLD também ficará vazio. Verifique o job Raw->Silver.")
        spark.stop()
        exit(0)
        
    df_silver.printSchema()
except AnalysisException as e:
    print(f"❌ Erro ao ler a tabela {SILVER_TABLE}. Verifique se a tabela existe e se as permissões estão corretas. Erro: {e}")
    spark.stop()
    exit(1)


Lendo tabela da Camada Silver: ResultadosCorridas


Aplicando CAST explícito nas colunas ID (chaves de negócio)...


✅ Dados da Silver carregados. Total de registros LIDOS: 319855
root
 |-- id_equipe: integer (nullable = true)
 |-- nome_equipe: string (nullable = true)
 |-- id_piloto: integer (nullable = true)
 |-- primeiro_nome_piloto: string (nullable = true)
 |-- sobrenome_piloto: string (nullable = true)
 |-- id_corrida: integer (nullable = true)
 |-- ano: integer (nullable = true)
 |-- rodada: integer (nullable = true)
 |-- nome_corrida: string (nullable = true)
 |-- id_status: integer (nullable = true)
 |-- descricao_status: string (nullable = true)
 |-- volta: integer (nullable = true)
 |-- posicao_na_volta: integer (nullable = true)
 |-- tempo_volta_ms: integer (nullable = true)
 |-- duracao_parada_seg: decimal(10,3) (nullable = true)
 |-- pontos_piloto: decimal(10,1) (nullable = true)
 |-- vitorias_piloto: integer (nullable = true)



### Seção 5: Criação e Carga das Dimensões (GOLD)


In [5]:
def save_dimension(df_dim, dim_name, id_col_silver, other_cols, cols_to_drop=None):
    """
    Salva dados na dimensão. A coluna ID_COL_SILVER é renomeada para CHV_..._ORG (Business Key)
    e as colunas de drop são removidas para evitar conflito com colunas geradas.
    """

    table_name = f"{GOLD_SCHEMA}.dim_{dim_name}"
    
    business_key_col = f"chv_{dim_name}_org"
    
    df_dim_unique = df_dim.select(
        col(id_col_silver).alias(business_key_col), 
        *other_cols
    ).distinct().dropna(subset=[business_key_col])

    if cols_to_drop:
        df_dim_unique = df_dim_unique.drop(*cols_to_drop)
    
    print(f"\n---> Criando e carregando Dimensão: {table_name}")
    count_unique = df_dim_unique.count()
    print(f"Número de registros únicos a serem inseridos: {count_unique}")
    
    if count_unique == 0:
        print("AVISO: DataFrame da Dimensão está vazio antes da escrita no banco!")
        return table_name

    try:
        df_dim_unique.write \
            .mode("append") \
            .jdbc(url=jdbc_url, table=table_name, properties=connection_properties)
        
        df_check = spark.read.jdbc(url=jdbc_url, table=table_name, properties=connection_properties)
        count_check = df_check.count()
        
        if count_check >= count_unique:
            print(f"✅ Dimensão {dim_name} carregada com sucesso! Registros CONFIRMADOS no PG: {count_check}")
        else:
            print(f"⚠️ AVISO DE CONTAGEM: O número de registros confirmados ({count_check}) é menor que o esperado ({count_unique}).")

    except Exception as e:
        print(f"❌ ERRO FATAL AO GRAVAR NA DIMENSÃO {table_name}: {e}")
        raise e
    
    return table_name

# --- Chamada 1: Piloto (dim_pil) ---
df_piloto_src = df_silver.select(
    "id_piloto", 
    col("primeiro_nome_piloto").alias("prim_nom"), 
    col("sobrenome_piloto").alias("sob_nom")       
).withColumn("nom_com", concat_ws(" ", col("prim_nom"), col("sob_nom"))) 

save_dimension(
    df_piloto_src, 
    "pil",                                  
    "id_piloto", 
    ["prim_nom", "sob_nom", "nom_com"],     
    cols_to_drop=["nom_com"]                
)

# --- Chamada 2: Equipe (dim_eqp) ---
df_eqp_src = df_silver.select(
    "id_equipe",
    col("nome_equipe").alias("nom_eqp")      
)

save_dimension(
    df_eqp_src,                             
    "eqp",                                  
    "id_equipe", 
    ["nom_eqp"]                             
)

# --- Chamada 3: Corrida (dim_cor) ---
df_cor_src = df_silver.select(
    "id_corrida",
    "ano",                                  
    col("rodada").alias("rod"),             
    col("nome_corrida").alias("nom_cor")    
)

save_dimension(
    df_cor_src,                             
    "cor",                                  
    "id_corrida", 
    ["ano", "rod", "nom_cor"]               
)

# --- Chamada 4: Status (dim_sts) ---
df_sts_src = df_silver.select(
    "id_status",
    col("descricao_status").alias("des_sts") 
)

save_dimension(
    df_sts_src,                             
    "sts",                                  
    "id_status", 
    ["des_sts"]                             
)


---> Criando e carregando Dimensão: gold.dim_pil


Número de registros únicos a serem inseridos: 77


✅ Dimensão pil carregada com sucesso! Registros CONFIRMADOS no PG: 77

---> Criando e carregando Dimensão: gold.dim_eqp


Número de registros únicos a serem inseridos: 23


✅ Dimensão eqp carregada com sucesso! Registros CONFIRMADOS no PG: 23

---> Criando e carregando Dimensão: gold.dim_cor


Número de registros únicos a serem inseridos: 286


✅ Dimensão cor carregada com sucesso! Registros CONFIRMADOS no PG: 286

---> Criando e carregando Dimensão: gold.dim_sts


Número de registros únicos a serem inseridos: 77


✅ Dimensão sts carregada com sucesso! Registros CONFIRMADOS no PG: 77


'gold.dim_sts'

### Seção 6: Criação e Carga da Tabela de Fato (FT_VOLTAS_TEMPO_PARADA)


In [6]:
print("\n\n=== Iniciando a construção da Tabela de Fato ===")

def read_dimension_with_srk(dim_abbr, silver_id_suffix):
    """
    Lê a Dimensão da Gold, pegando SRK e Chave de Negócio.
    A Chave de Negócio (chv_..._org) é RENOMEADA para id_... (ex: id_piloto) 
    para o JOIN com a tabela Silver.
    """
    table_name = f"{GOLD_SCHEMA}.dim_{dim_abbr}"
    
    srk_col = f"srk_{dim_abbr}"
    
    chave_origem_col = f"chv_{dim_abbr}_org" 
    
    df_dim = spark.read.jdbc(url=jdbc_url, table=table_name, properties=connection_properties)

    return df_dim.select(col(srk_col), col(chave_origem_col).alias(f"id_{silver_id_suffix}"))

df_dim_piloto   = read_dimension_with_srk("pil", "piloto")
df_dim_equipe   = read_dimension_with_srk("eqp", "equipe")
df_dim_corrida  = read_dimension_with_srk("cor", "corrida")
df_dim_status   = read_dimension_with_srk("sts", "status")

print(f"DEBUG: SRKs de Piloto lidas do Gold: {df_dim_piloto.count()}")

df_fato_base = df_silver.alias("S") \
    .join(df_dim_piloto.alias("DP"), col("S.id_piloto") == col("DP.id_piloto"), "inner") \
    .join(df_dim_equipe.alias("DE"), col("S.id_equipe") == col("DE.id_equipe"), "inner") \
    .join(df_dim_corrida.alias("DC"), col("S.id_corrida") == col("DC.id_corrida"), "inner") \
    .join(df_dim_status.alias("DS"), col("S.id_status") == col("DS.id_status"), "inner")

count_fato_base = df_fato_base.count()
print(f"DEBUG: Registros após todos os JOINs: {count_fato_base}")
if count_fato_base == 0:
    print("❌ ERRO CRÍTICO: Zero registros após JOINs! As chaves de negócio (IDs) da Silver não bateram com as chaves de origem da Gold. Verifique o DDL Gold (UNIQUE) e a tipagem.")
    spark.stop()
    exit(1)


df_fato = df_fato_base.select(
    col("DP.srk_pil").alias("srk_pil"),     
    col("DE.srk_eqp").alias("srk_eqp"),     
    col("DC.srk_cor").alias("srk_cor"),     
    col("DS.srk_sts").alias("srk_sts"),     
    
    col("S.volta").alias("volt").cast(IntegerType()),                   
    col("S.posicao_na_volta").alias("pos_volt").cast(IntegerType()),    
    col("S.tempo_volta_ms").alias("tmp_volt_ms").cast(IntegerType()),   
    col("S.duracao_parada_seg").alias("dur_par_seg").cast(DecimalType(10, 3)), 
    
    col("S.pontos_piloto").alias("pnt_pil").cast(DecimalType(10, 1)),   
    col("S.vitorias_piloto").alias("vit_pil").cast(IntegerType())       
)


df_fato = df_fato.filter(col("volt").isNotNull() & col("tmp_volt_ms").isNotNull())

print("\nEstrutura final da Tabela de Fato:")
df_fato.printSchema()
print(f"Total de registros na Fato (após filtro): {df_fato.count()}")


FACT_TABLE = f"{GOLD_SCHEMA}.fat_des_volt"
print(f"\nIniciando carga na tabela de Fato: {FACT_TABLE}")

try:
    df_fato.write \
        .mode("overwrite") \
        .jdbc(url=jdbc_url, table=FACT_TABLE, properties=connection_properties)
            
    print(f"✅ Tabela de Fato '{FACT_TABLE}' populada com sucesso!")
except Exception as e:
    print(f"❌ ERRO FATAL AO GRAVAR NA TABELA DE FATO {FACT_TABLE}: {e}")
    raise e



=== Iniciando a construção da Tabela de Fato ===


DEBUG: SRKs de Piloto lidas do Gold: 77


DEBUG: Registros após todos os JOINs: 319855

Estrutura final da Tabela de Fato:
root
 |-- srk_pil: integer (nullable = true)
 |-- srk_eqp: integer (nullable = true)
 |-- srk_cor: integer (nullable = true)
 |-- srk_sts: integer (nullable = true)
 |-- volt: integer (nullable = true)
 |-- pos_volt: integer (nullable = true)
 |-- tmp_volt_ms: integer (nullable = true)
 |-- dur_par_seg: decimal(10,3) (nullable = true)
 |-- pnt_pil: decimal(10,1) (nullable = true)
 |-- vit_pil: integer (nullable = true)



Total de registros na Fato (após filtro): 319683

Iniciando carga na tabela de Fato: gold.fat_des_volt


✅ Tabela de Fato 'gold.fat_des_volt' populada com sucesso!


### Seção 7: Finalização

In [7]:
print("\n🚀 Job ETL (Silver → Gold) finalizado com sucesso!")
spark.stop()


🚀 Job ETL (Silver → Gold) finalizado com sucesso!
